# ⛵️ 드리프트 도입

![trusty-drift-meme.png](../images/trusty-drift-meme.png)

이제 TrustyAI를 드리프트 탐지하도록 설정했으므로, 몇 가지를 도입해 보고 어떤 일이 일어나는지 봅시다.  
이를 위해 단순히 테스트 데이터셋을 기반으로 많은 요청을 보낼 것입니다.  
TrustyAI에 학습 데이터셋의 작은 크기만 보냈기 때문에 테스트 데이터셋이 상당히 다르게 보일 가능성이 높습니다.

In [ ]:
!pip -q install onnxruntime model-registry==0.2.15

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import numpy as np
import requests
from fetch_artifacts_from_registry import fetch_artifacts_from_registry

In [ ]:
# 일부 경고 메시지 비활성화
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

여기에 테스트 모델 엔드포인트를 `infer_endpoint`로 추가하세요.

In [ ]:
infer_endpoint = "ENTER-YOUR-INFERENCE-ENDPOINT"

deployed_model_name = "jukebox"

In [ ]:
infer_url = f"{infer_endpoint}/v2/models/{deployed_model_name}/infer"

In [ ]:
def rest_request(data):
    json_data = {
        "inputs": [
           {
                "name": name,
                "shape": [1, 1],
                "datatype": "FP32",
                "data": [data[name]]
            }
            for name in data.keys()
        ]
    }
    response = requests.post(infer_url, json=json_data, verify=True)
    response_dict = response.json()
    return response_dict['outputs'][0]['data']


In [ ]:
# 테스트 데이터 로드
X_test = pd.read_pickle("preprocess-data/test_data.pkl")[0][:500]

우리는 테스트 데이터셋에서 500개 샘플을 보내고 있으므로 이것은 몇 분 정도 걸릴 수 있습니다.

In [ ]:
# 예측값 전송
i=1
for _, row in X_test.iterrows():
    if i%100==0:
        print(f"{i} samples sent")
    prediction = rest_request(row.to_dict())
    i+=1

이제 지시사항으로 돌아갈 시간입니다 🏃💨